# Colab 전용 - LoRA 학습 + 자동 업로드

- 데이터: `finetune_dataset.jsonl` (`instruction` / `input` / `output`)
- 학습 스크립트: `finetune/train_lora.py`
- 업로드 로직: 노트북 셀 내부 내장(`huggingface_hub` 직접 호출)

학습이 끝나면 `ENABLE_UPLOAD=True`일 때 자동으로 Hugging Face에 업로드합니다.

In [ ]:
import os

# 프로젝트를 /content/metadata 로 업로드/clone 했다고 가정
PROJECT_ROOT = "/content/metadata"
FINETUNE_DIR = f"{PROJECT_ROOT}/finetune"
DATA_PATH = "/content/finetune_dataset.jsonl"
OUTPUT_DIR = "/content/lora-output"
BASE_MODEL = "MLP-KTLim/llama-3-Korean-Bllossom-8B"
USE_4BIT = True  # T4 권장: True, L4 여유 있으면 False도 가능

# 자동 업로드 옵션
ENABLE_UPLOAD = True
HF_REPO_ID = "seongsu0105/blossom-8b-adapter"
HF_REPO_PRIVATE = False

In [ ]:
!pip install -q -r /content/metadata/finetune/requirements-train.txt
!pip install -q python-dotenv huggingface_hub

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

script = Path(FINETUNE_DIR).resolve() / "train_lora.py"
if not script.is_file():
    raise FileNotFoundError(f"train_lora.py 없음: {script}")

cmd = [
    sys.executable,
    "-u",
    str(script),
    "--data", DATA_PATH,
    "--out", OUTPUT_DIR,
    "--base-model", BASE_MODEL,
    "--epochs", "2",
    "--batch-size", "2",
    "--grad-accum", "4",
    "--max-length", "2048",
]
if USE_4BIT:
    cmd.append("--use-4bit")

tail = deque(maxlen=400)
proc = subprocess.Popen(
    cmd,
    cwd=str(script.parent),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    tail.append(line)
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError("".join(tail))

print(f"학습 완료: {OUTPUT_DIR}")

In [ ]:
if ENABLE_UPLOAD:
    from google.colab import userdata
    from huggingface_hub import HfApi, login

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Colab Secrets에 HF_TOKEN을 먼저 등록하세요.")

    if not os.path.isdir(OUTPUT_DIR):
        raise RuntimeError(f"업로드 폴더가 없습니다: {OUTPUT_DIR}")

    login(token=token)
    api = HfApi()
    api.create_repo(
        repo_id=HF_REPO_ID,
        repo_type="model",
        private=HF_REPO_PRIVATE,
        exist_ok=True,
        token=token,
    )
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        token=token,
    )
    print(f"허깅페이스 자동 업로드 완료: https://huggingface.co/{HF_REPO_ID}")
else:
    print("ENABLE_UPLOAD=False 이므로 업로드를 건너뜁니다.")